# Du marché au bilan : le pont théorie des jeux ↔ théorie de la décision

> **Navigation** : [GameTheory-17](GameTheory-17-MultiAgent-RL-Python.ipynb) ·
> [GameTheory-17b](GameTheory-17b-Asymmetric-Information-Python.ipynb) (prérequis direct) ·
> ce carnet est la **face assureur** de 17b.

[GameTheory-17b](GameTheory-17b-Asymmetric-Information-Python.ipynb) a établi la **vue
marché** de l'information asymétrique : qui contracte avec qui, à quel prix
d'équilibre — point fixe d'Akerlof, signal de Spence, menu séparateur de
Rothschild–Stiglitz, anticipation de Wilson/Miyazaki. Ce carnet relit **les
mêmes objets depuis l'autre côté du comptoir**, celui de l'assureur qui doit :

1. **tarifer** — décomposer la prime en prime pure et chargement (§1-§2) ;
2. **créditer l'expérience individuelle** — crédibilité de Bühlmann–Straub (§3) ;
3. **immobiliser du capital** — ruine de Cramér–Lundberg quand l'équilibre de
   marché rate (§4) ;
4. **payer pour de l'information** — la valeur d'un questionnaire de
   souscription en équivalent EVSI (§5).

La correspondance est exacte, pas une métaphore : le contrat $(\alpha, \beta)$
de 17b (couverture $\alpha$, prime $\beta$) est l'objet unique lu deux fois —
comme échange d'utilité espérée côté marché, comme flux de primes et de
sinistres côté bilan. Chaque section croise ainsi un modèle de 17b avec une
tranche de la série actuarielle *DecisionTheory* (prime, crédibilité, ruine,
valeur d'information — carnets DecPyMC référencés par concept).

**Prérequis** : 17b (contrats, types, menus) ; notions de Poisson et
d'espérance conditionnelle.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

// Les deux types de risque de la serie Decision Theory (T1) :
// frequence de sinistre annuelle, severite unitaire.
const double THETA_LOW = 0.01, THETA_HIGH = 0.04;
const double GAMMA = 0.7; // aversion au risque absolue (CARA), comme en 17b
const double WEALTH = 10.0; // richesse initiale de l'assure
const double SEVERITY = 1.0; // sinistralite unitaire (indemnite = alpha * severite)

// Comparaison à une référence affichée, avec tolérance explicite.
var ecarts = new Dictionary<string, double>();
void Verifier(string section, double obtenu, double attendu, double tolerance)
{
    double ecart = Math.Abs(obtenu - attendu);
    if (double.IsNaN(obtenu) || double.IsInfinity(obtenu) || ecart > tolerance)
        throw new Exception($"{section} : obtenu {obtenu}, attendu {attendu}, écart {ecart}, tolérance {tolerance}");
    if (!ecarts.ContainsKey(section) || ecart > ecarts[section]) ecarts[section] = ecart;
}

## §1 Le contrat, objet double : utilité espérée contre flux de sinistres

**Côté marché** (17b) : l'assuré de type $\theta$ compare les contrats par
utilité espérée CARA, $u(w) = -e^{-\gamma w}$ ; un menu est un équilibre
séparateur si chaque type préfère *son* contrat (incitation) et préfère
s'assurer plutôt que de s'arrêter (participation).

**Côté assureur** : le même contrat $(\alpha, \beta)$ promet une prime
$\beta$ encaissée sûrement et une indemnité $\alpha$ versée avec probabilité
$\theta$. Le **profit espéré** est $\beta - \theta\alpha$ : la **prime pure**
$\theta\alpha$ est le point d'équilibre actuariel, et la concurrence pure de
Rothschild–Stiglitz pousse précisément chaque contrat séparateur sur cette
ligne — **le séparateur RS est un menu break-even par type**.

In [2]:
// Le même objet qu'en 17b : couverture alpha, prime beta.
public record Contract(double Coverage, double Premium)
{
    public double ExpectedLoss(double theta) => theta * Coverage * 1.0;
    public double ExpectedProfit(double theta) => Premium - ExpectedLoss(theta);
}

double CaraUtilityWealth(double wealth) => -Math.Exp(-GAMMA * wealth);
double ExpectedUtility(Contract c, double theta)
{
    double wNoLoss = WEALTH - c.Premium;
    double wLoss = WEALTH - c.Premium + c.Coverage * SEVERITY;
    return (1 - theta) * CaraUtilityWealth(wNoLoss) + theta * CaraUtilityWealth(wLoss);
}
double OutsideOption(double theta) =>
    (1 - theta) * CaraUtilityWealth(WEALTH) + theta * CaraUtilityWealth(WEALTH - SEVERITY);
var OUTSIDE_OPTION_U = new Dictionary<double, double>
{
    [THETA_LOW] = OutsideOption(THETA_LOW),
    [THETA_HIGH] = OutsideOption(THETA_HIGH)
};

// Menu separateur RS : pleine couverture break-even pour chaque type.
var menu_rs = new[]
{
    (Label: "L (risque faible)", Theta: THETA_LOW, Contrat: new Contract(1.0, THETA_LOW), EU: -0.00091, Outside: -0.00092),
    (Label: "H (risque fort)", Theta: THETA_HIGH, Contrat: new Contract(1.0, THETA_HIGH), EU: -0.00092, Outside: -0.00095)
};
Console.WriteLine("contrat | prime β | sinistre espéré θα | profit espéré | EU(assuré) | option extérieure");
foreach (var row in menu_rs)
{
    double loss = row.Contrat.ExpectedLoss(row.Theta);
    double profit = row.Contrat.ExpectedProfit(row.Theta);
    double eu = ExpectedUtility(row.Contrat, row.Theta);
    double outside = OUTSIDE_OPTION_U[row.Theta];
    // Les utilités de référence sont arrondies à cinq décimales dans la source.
    Verifier("contrat", row.Contrat.Premium, row.Theta, 1e-12);
    Verifier("contrat", loss, row.Theta, 1e-12);
    Verifier("contrat", profit, 0.0, 1e-12);
    Verifier("contrat", eu, row.EU, 0.000005);
    Verifier("contrat", outside, row.Outside, 0.000005);
    Console.WriteLine($"{row.Label} | {row.Contrat.Premium:F2} | {loss:F2} | {profit:F1} | {eu:F5} | {outside:F5}");
}

contrat | prime β | sinistre espéré θα | profit espéré | EU(assuré) | option extérieure


L (risque faible) | 0,01 | 0,01 | 0,0 | -0,00091 | -0,00092


H (risque fort) | 0,04 | 0,04 | 0,0 | -0,00092 | -0,00095


### Lecture du résultat

Chaque contrat séparateur est exactement break-even ($\beta = \theta\alpha$,
profit espéré nul) : l'hypothèse de concurrence pure **transfère tout le
surplus à l'assuré**. Un assureur réel ne peut pas vivre à l'équilibre
concurrentiel — il doit ajouter un **chargement** $\eta$ sur la prime pure,
$\beta = (1+\eta)\,\theta\alpha$, et ce chargement est immédiatement
plafonné par la contrainte de participation : trop cher, l'assuré retourne à
son option extérieure et le contrat ne vend plus. C'est la décomposition
**prime pure / chargement** de la tranche T1, rencontrée ici depuis le
mécanisme de marché.

## §2 Le chargement maximal : ce que la participation tolère

Le chargement de sécurité le plus élevé que chaque type accepte est celui qui
ramène son utilité espérée exactement à son option extérieure (ne pas
s'assurer). On le trouve par dichotomie : c'est le prix de réserve **du côté
de la demande**, la borne supérieure de tout tarif réel.

In [3]:
double MaxLoading(double theta)
{
    double EuGap(double eta) => ExpectedUtility(new Contract(1.0, (1.0 + eta) * theta), theta) - OUTSIDE_OPTION_U[theta];
    double lo = 0.0, hi = 50.0 / theta; // borne large : au-dela, la prime ruine l'assure
    if (EuGap(hi) > 0) return hi;
    for (int i = 0; i < 80; i++)
    {
        double mid = 0.5 * (lo + hi);
        if (EuGap(mid) > 0) lo = mid;
        else hi = mid;
    }
    return 0.5 * (lo + hi);
}
var eta_max = new Dictionary<double, double>
{
    [THETA_LOW] = MaxLoading(THETA_LOW),
    [THETA_HIGH] = MaxLoading(THETA_HIGH)
};
var loadingRows = new[]
{
    (Label: "L (θ=0.01)", Theta: THETA_LOW, EtaReference: 1.1619, PrimeReference: 0.0216),
    (Label: "H (θ=0.04)", Theta: THETA_HIGH, EtaReference: 1.1461, PrimeReference: 0.0858)
};
Console.WriteLine("type | θ | prime pure θα | chargement max η* | prime max (1+η*)θ");
foreach (var row in loadingRows)
{
    double eta = eta_max[row.Theta];
    double prime = (1 + eta) * row.Theta;
    // Les références sont arrondies à quatre décimales dans la source.
    Verifier("chargement", eta, row.EtaReference, 0.00005);
    Verifier("chargement", prime, row.PrimeReference, 0.00005);
    Verifier("participation", ExpectedUtility(new Contract(1.0, prime), row.Theta), OUTSIDE_OPTION_U[row.Theta], 1e-12);
    Console.WriteLine($"{row.Label} | {row.Theta:F2} | {row.Theta:F2} | {eta:F4} | {prime:F4}");
}

type | θ | prime pure θα | chargement max η* | prime max (1+η*)θ


L (θ=0.01) | 0,01 | 0,01 | 1,1619 | 0,0216


H (θ=0.04) | 0,04 | 0,04 | 1,1461 | 0,0858


### Lecture du résultat

Surprise : les chargements maximaux **relatifs** sont presque identiques pour
les deux types (à peine plus d'une fois la prime pure), alors que les primes
maximales **absolues** diffèrent d'un facteur quatre. Avec une aversion CARA
et de petites fréquences, la disposition à payer est à peu près
proportionnelle au risque — ce n'est pas le type qui structure le chargement,
c'est **l'aversion au risque** $\gamma$ (exercice : refaire le tableau avec
$\gamma$ doublé). Le chargement réel d'un assureur doit donc rester sous un
plafond **homogène** ~1,15·θ dans ce marché : peu de marge, et toute la
question devient *avec quelle information* l'assureur peut-il prétendre
connaître le $\theta$ auquel ce plafond s'applique — c'est la crédibilité.

## §3 Crédibilité : le marché vend un prior, l'assureur achète de l'expérience

Le menu séparateur suppose le type **connu** — information parfaite. Le
pooling suppose le type **inconnu et identique pour tous**. L'assureur réel
vit entre les deux : il observe un historique de sinistres. La crédibilité de
Bühlmann–Straub quantifie ce chemin :

$$Z = \frac{n}{n + K}, \qquad K = \frac{\text{EPV}}{\text{VHM}},$$

où $n$ est le nombre d'années observées, EPV la variance intra-individu du
comptage et VHM la variance inter-individus des fréquences moyennes. Pour un
comptage **Poisson** de mélange binomial, EPV admet l'estimateur fermé
$s^2 = m$ (la variance conditionnelle d'une Poisson est sa moyenne) — ne pas
la confondre avec la VHM, erreur classique qui produit un $K$ minuscule et un
$Z \approx 1$, c'est-à-dire du no-pooling déguisé.

In [4]:
// Constantes de Bühlmann–Straub pour un mélange de deux lois de Poisson.
(double Epv, double Vhm, double K) BuhlmannStraubConstants(double piHigh, double lamLow, double lamHigh)
{
    double m = piHigh * lamHigh + (1 - piHigh) * lamLow;
    double vhm = piHigh * (1 - piHigh) * Math.Pow(lamHigh - lamLow, 2);
    return (m, vhm, m / vhm);
}
const double PI_HIGH = 0.30;
var constants = BuhlmannStraubConstants(PI_HIGH, THETA_LOW, THETA_HIGH);
double EPV = constants.Epv, VHM = constants.Vhm, K = constants.K;
double n_star_half = K * 0.5 / (1 - 0.5);
double n_star_ninety = K * 0.9 / (1 - 0.9);
Verifier("crédibilité", EPV, 0.019, 1e-12);
Verifier("crédibilité", VHM, 0.000189, 1e-12);
Verifier("crédibilité", K, 0.019 / 0.000189, 1e-10);
Verifier("crédibilité", n_star_half / (n_star_half + K), 0.5, 1e-12);
Verifier("crédibilité", n_star_ninety / (n_star_ninety + K), 0.9, 1e-12);

var annees = Enumerable.Range(0, 121).ToArray();
var zValues = annees.Select(n => n / (n + K)).ToArray();
var referenceZ = new[] { (1, 0.010), (3, 0.029), (5, 0.047), (10, 0.090), (30, 0.230) };
Console.WriteLine("années observées n | Z | lecture");
foreach (var (n, reference) in referenceZ)
{
    double z = zValues[n];
    Verifier("crédibilité arrondie", z, reference, 0.0005);
    string lecture = z < 0.3 ? "proche du pooling" : z < 0.7 ? "mi-chemin" : "proche du séparateur";
    Console.WriteLine(FormattableString.Invariant($"{n} | {z:F3} | {lecture}"));
}

// Le formateur HTML émet le SVG lui-même, sans l'échapper en texte.
public record Graphique(string Svg);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<Graphique>(
    (g, writer) => writer.Write(g.Svg), "text/html");
var credSvg = new StringBuilder();
credSvg.Append("<svg xmlns='http://www.w3.org/2000/svg' width='700' height='360' viewBox='0 0 700 360' role='img' aria-label='Crédibilité Bühlmann–Straub'>");
credSvg.Append("<rect width='700' height='360' fill='white'/><text x='350' y='25' text-anchor='middle'>Crédibilité Bühlmann–Straub (mélange 70% L / 30% H)</text>");
credSvg.Append("<path d='M70 40 V300 H660' fill='none' stroke='black'/><path d='M70 170 H660' stroke='red' stroke-dasharray='6 4'/><path d='M70 40 H660' stroke='grey' stroke-dasharray='2 4'/>");
string credPoints = string.Join(" ", annees.Select(n => FormattableString.Invariant($"{70.0 + 590.0 * n / 120.0:F2},{300.0 - 260.0 * zValues[n]:F2}")));
credSvg.Append($"<polyline points='{credPoints}' fill='none' stroke='steelblue' stroke-width='2'/>");
double starX = 70 + 590 * n_star_half / 120;
credSvg.Append(FormattableString.Invariant($"<circle cx='{starX:F2}' cy='170' r='4' fill='red'/><text x='{starX - 110:F2}' y='215'>n* = {n_star_half:F0} ans</text>"));
credSvg.Append(FormattableString.Invariant($"<text x='80' y='60'>Z = n/(n+K), K = {K:F1}</text><text x='80' y='162'>Z = 0.5</text><text x='350' y='340' text-anchor='middle'>années d'historique n</text><text x='15' y='185' transform='rotate(-90 15 185)'>facteur de crédibilité Z</text><text x='60' y='315'>0</text><text x='640' y='315'>120</text></svg>"));
display(new Graphique(credSvg.ToString()));

années observées n | Z | lecture


1 | 0.010 | proche du pooling


3 | 0.029 | proche du pooling


5 | 0.047 | proche du pooling


10 | 0.090 | proche du pooling


30 | 0.230 | proche du pooling


Crédibilité Bühlmann–Straub (mélange 70% L / 30% H) <polyline points='70.00,300.00 74.92,297.44 79.83,294.93 84.75,292.47 89.67,290.05 94.58,287.68 99.50,285.36 104.42,283.07 109.33,280.83 114.25,278.64 119.17,276.48 124.08,274.36 129.00,272.27 133.92,270.23 138.83,268.22 143.75,266.24 148.67,264.30 153.58,262.39 158.50,260.52 163.42,258.67 168.33,256.86 173.25,255.07 178.17,253.32 183.08,251.59 188.00,249.89 192.92,248.22 197.83,246.57 202.75,244.95 207.67,243.36 212.58,241.79 217.50,240.24 222.42,238.72 227.33,237.22 232.25,235.74 237.17,234.29 242.08,232.86 247.00,231.44 251.92,230.05 256.83,228.68 261.75,227.33 266.67,225.99 271.58,224.68 276.50,223.38 281.42,222.11 286.33,220.85 291.25,219.60 296.17,218.38 301.08,217.17 306.00,215.98 310.92,214.80 315.83,213.64 320.75,212.49 325.67,211.36 330.58,210.25 335.50,209.14 340.42,208.06 345.33,206.98 350.25,205.92 355.17,204.88 360.08,203.84 365.00,202.82 369.92,201.81 374.83,200.82 379.75,199.83 384.67,198.86 389.58,197.90 394.50,196.95 399.42,196.02 404.33,195.09 409.25,194.18 414.17,193.27 419.08,192.38 424.00,191.50 428.92,190.62 433.83,189.76 438.75,188.91 443.67,188.06 448.58,187.23 453.50,186.41 458.42,185.59 463.33,184.78 468.25,183.99 473.17,183.20 478.08,182.42 483.00,181.64 487.92,180.88 492.83,180.13 497.75,179.38 502.67,178.64 507.58,177.91 512.50,177.18 517.42,176.47 522.33,175.76 527.25,175.06 532.17,174.36 537.08,173.68 542.00,173.00 546.92,172.32 551.83,171.66 556.75,171.00 561.67,170.34 566.58,169.70 571.50,169.06 576.42,168.42 581.33,167.79 586.25,167.17 591.17,166.56 596.08,165.95 601.00,165.34 605.92,164.74 610.83,164.15 615.75,163.56 620.67,162.98 625.58,162.41 630.50,161.84 635.42,161.27 640.33,160.71 645.25,160.16 650.17,159.61 655.08,159.06 660.00,158.52' fill='none' stroke='steelblue' stroke-width='2'/> n* = 101 ans Z = n/(n+K), K = 100.5 Z = 0.5 années d'historique n facteur de crédibilité Z 0 120

### Lecture du résultat

Lire $K$ dans la sortie : de l'ordre de **cent** années, $n^*$ (Z = 0.5) aussi
— après 30 ans d'historique, $Z$ plafonne à 0.23. Le comptage Poisson annuel
est un signal **très bruité** : avec des fréquences de 1% à 4%, la plupart
des années sans sinistre n'apprennent presque rien sur le type. La
crédibilité mesure ici le coût en **temps** de l'information — et ce temps
excède une carrière d'assureur. Le menu séparateur de 17b correspond à $Z = 1$
**immédiat** ; le questionnaire de souscription (§5) achète en une transaction
une grande partie de ce qu'un siècle d'historique ne construirait que
lentement : voilà pourquoi la sélection à l'entrée domine l'*experience
rating* sur ce marché — la lecture assureur du problème d'antisélection.

## §4 Ruine : quand l'équilibre de marché rate, c'est le bilan qui paie

17b montre qu'un équilibre Rothschild–Stiglitz **peut ne pas exister** : selon
la proportion de types H, un contrat déviant détruit le menu. La chaîne
exacte, côté assureur, est l'**écrémage** :

1. **Pooling** : l'assureur propose un contrat unique au prix moyen
   $\bar\theta = \pi_H\theta_H + (1-\pi_H)\theta_L$ — break-even si le
   mélange reste équilibré.
2. **Écrémage** : un déviant propose aux types L un contrat
   $(1,\,\theta_L + \varepsilon)$ strictement meilleur pour eux — le
   pooling ne peut pas suivre sans perdre les H. Les L partent.
3. **Spirale** : le portefeuille restant ne contient plus que des H, mais la
   prime est toujours $\bar\theta < \theta_H$. Chaque contrat devient
   perdant en espérance : le processus de surplus de Cramér–Lundberg

$$U(t) = u + \sum_i \beta_i\, t - S(t)$$

dérive vers la **ruine**. Le menu séparateur, lui, rétablit le break-even par
type. Simulons les deux destins — l'étape 2 étant jouée, le portefeuille A
n'a plus que des H à la prime moyenne ; le portefeuille B reprime chaque type
à sa propre fréquence, plus un chargement de sécurité.

In [5]:
// Surplus discrétisé : le générateur .NET ne reproduit pas les tirages NumPy.
public record PortfolioResult(double RuinFrequency, double FinalSurvivorsMean, double[,] Paths);
int Poisson(Random random, double lambda)
{
    double limit = Math.Exp(-lambda), product = 1.0;
    int count = 0;
    do { count++; product *= random.NextDouble(); } while (product > limit);
    return count - 1;
}
PortfolioResult SimulatePortfolio(int nPolicyholders, double premiumPerHead, double claimRatePerHead,
    double loading, double horizonYears = 10.0, int nPaths = 400, double initialCapital = 4.0, int seed = 17)
{
    var gen = new Random(seed);
    double dt = 1.0 / 12.0;
    int steps = (int)(horizonYears / dt);
    double premiumFlow = nPolicyholders * premiumPerHead * (1 + loading) * dt;
    double lamStep = nPolicyholders * claimRatePerHead * dt;
    var surplus = Enumerable.Repeat(initialCapital, nPaths).ToArray();
    var alive = Enumerable.Repeat(true, nPaths).ToArray();
    var paths = new double[steps + 1, nPaths];
    for (int i = 0; i < nPaths; i++) paths[0, i] = surplus[i];
    for (int t = 1; t <= steps; t++)
        for (int i = 0; i < nPaths; i++)
        {
            double claims = Poisson(gen, lamStep) * SEVERITY;
            double next = surplus[i] + premiumFlow - claims;
            alive[i] = alive[i] && next >= 0;
            if (alive[i]) surplus[i] = next;
            paths[t, i] = surplus[i];
        }
    int survivors = alive.Count(a => a);
    double mean = survivors == 0 ? double.NaN : Enumerable.Range(0, nPaths).Where(i => alive[i]).Average(i => surplus[i]);
    return new PortfolioResult((double)(nPaths - survivors) / nPaths, mean, paths);
}

double pool_mix_mean = PI_HIGH * THETA_HIGH + (1 - PI_HIGH) * THETA_LOW;
var portfolio_pooling = SimulatePortfolio(200, pool_mix_mean, THETA_HIGH, 0.0, seed: 17);
int n_h = (int)(200 * PI_HIGH), n_l = (int)(200 * (1 - PI_HIGH));
double sep_mean_rate = (n_h * THETA_HIGH + n_l * THETA_LOW) / 200;
var portfolio_separating = SimulatePortfolio(200, sep_mean_rate, sep_mean_rate, 0.30, seed: 17);
Verifier("portefeuilles exacts", pool_mix_mean, 0.019, 1e-12);
Verifier("portefeuilles exacts", (pool_mix_mean - THETA_HIGH) * 200 / 12, -0.350, 1e-12);
Verifier("portefeuilles exacts", sep_mean_rate * 1.30, 0.0247, 1e-12);
Verifier("portefeuilles exacts", sep_mean_rate * 0.30 * 200 / 12, 0.095, 1e-12);

// Les références suivantes sont citées de la sortie Python, non recalculées ici.
// Tolérances de comparaison des simulations distinctes : 0.06 pour une fréquence,
// 1.00 pour un surplus. Elles ne constituent pas une égalité des trajectoires.
const double toleranceFrequence = 0.06, toleranceSurplus = 1.00;
Verifier("simulation A : fréquence", portfolio_pooling.RuinFrequency, 1.000, toleranceFrequence);
Verifier("simulation B : fréquence", portfolio_separating.RuinFrequency, 0.077, toleranceFrequence);
Verifier("simulation B : surplus", portfolio_separating.FinalSurvivorsMean, 16.76, toleranceSurplus);
if (!double.IsNaN(portfolio_pooling.FinalSurvivorsMean))
    throw new Exception("La référence Python citée pour le surplus A est NaN : comparer séparément les survivants.");
Console.WriteLine("Les tirages NumPy ne sont pas disponibles : simulation .NET distincte, graine 17 ; références Python citées ci-dessous.");
Console.WriteLine("portefeuille | prime par tête | drift mensuel | fréquence de ruine .NET | surplus final .NET");
Console.WriteLine(FormattableString.Invariant($"A pooling écrémé | {pool_mix_mean:F4} | {(pool_mix_mean - THETA_HIGH) * 200 / 12:F3} | {portfolio_pooling.RuinFrequency:F3} | {portfolio_pooling.FinalSurvivorsMean:F2}"));
Console.WriteLine(FormattableString.Invariant($"B séparateur + η=30% | {sep_mean_rate * 1.30:F4} | {sep_mean_rate * 0.30 * 200 / 12:F3} | {portfolio_separating.RuinFrequency:F3} | {portfolio_separating.FinalSurvivorsMean:F2}"));
Console.WriteLine("Références Python citées : A ruine 1.000, surplus NaN ; B ruine 0.077, surplus 16.76.");
Console.WriteLine(FormattableString.Invariant($"Écarts .NET − référence Python : ruine A {portfolio_pooling.RuinFrequency - 1.000:F3}, ruine B {portfolio_separating.RuinFrequency - 0.077:F3}, surplus B {portfolio_separating.FinalSurvivorsMean - 16.76:F2} ; surplus A non comparable (NaN)."));
Console.WriteLine(FormattableString.Invariant($"Tolérances absolues déclarées : fréquence {toleranceFrequence:F2}, surplus {toleranceSurplus:F2}."));

// Trajectoires .NET calculées, distinctes des trajectoires NumPy.
var plot = new StringBuilder();
plot.Append("<svg xmlns='http://www.w3.org/2000/svg' width='1100' height='380' viewBox='0 0 1100 380' role='img' aria-label='Trajectoires simulées des portefeuilles'><rect width='1100' height='380' fill='white'/>");
var portfolios = new[] { portfolio_pooling, portfolio_separating };
var colors = new[] { "steelblue", "darkorange", "forestgreen", "purple" };
var indices = new[] { 0, 7, 42, 133 };
double maxSurplus = portfolios.SelectMany(p => indices.SelectMany(i => Enumerable.Range(0, p.Paths.GetLength(0)).Select(t => p.Paths[t, i]))).Max();
double yMax = Math.Max(1.0, maxSurplus);
for (int panel = 0; panel < 2; panel++)
{
    double left = 65 + panel * 535;
    plot.Append(FormattableString.Invariant($"<path d='M{left:F1} 45 V310 H{left + 470:F1}' fill='none' stroke='black'/>"));
    string title = panel == 0 ? "A pooling écrémé : prime moyenne, H restants" : "B séparateur : prime par type + chargement 30%";
    plot.Append(FormattableString.Invariant($"<text x='{left + 235:F1}' y='25' text-anchor='middle'>{title}</text>"));
    for (int j = 0; j < indices.Length; j++)
    {
        int pathIndex = indices[j];
        int currentPanel = panel;
        string points = string.Join(" ", Enumerable.Range(0, portfolios[panel].Paths.GetLength(0)).Select(t => FormattableString.Invariant($"{left + 470.0 * t / 120:F2},{310.0 - 260.0 * portfolios[currentPanel].Paths[t, pathIndex] / yMax:F2}")));
        plot.Append($"<polyline points='{points}' fill='none' stroke='{colors[j]}' stroke-width='1.2'/>");
    }
    plot.Append(FormattableString.Invariant($"<text x='{left + 235:F1}' y='350' text-anchor='middle'>années</text><text x='{left:F1}' y='328'>0</text><text x='{left + 455:F1}' y='328'>10</text>"));
}
plot.Append("<text x='15' y='200' transform='rotate(-90 15 200)'>surplus U(t)</text></svg>");
display(new Graphique(plot.ToString()));

Les tirages NumPy ne sont pas disponibles : simulation .NET distincte, graine 17 ; références Python citées ci-dessous.


portefeuille | prime par tête | drift mensuel | fréquence de ruine .NET | surplus final .NET


A pooling écrémé | 0.0190 | -0.350 | 1.000 | NaN


B séparateur + η=30% | 0.0247 | 0.095 | 0.128 | 15.95


Références Python citées : A ruine 1.000, surplus NaN ; B ruine 0.077, surplus 16.76.


Écarts .NET − référence Python : ruine A 0.000, ruine B 0.051, surplus B -0.81 ; surplus A non comparable (NaN).


Tolérances absolues déclarées : fréquence 0.06, surplus 1.00.


A pooling écrémé : prime moyenne, H restants <polyline points='65.00,256.39 68.92,265.55 72.83,261.31 76.75,270.46 80.67,279.62 84.58,275.38 88.50,284.54 92.42,307.10 96.33,307.10 100.25,307.10 104.17,307.10 108.08,307.10 112.00,307.10 115.92,307.10 119.83,307.10 123.75,307.10 127.67,307.10 131.58,307.10 135.50,307.10 139.42,307.10 143.33,307.10 147.25,307.10 151.17,307.10 155.08,307.10 159.00,307.10 162.92,307.10 166.83,307.10 170.75,307.10 174.67,307.10 178.58,307.10 182.50,307.10 186.42,307.10 190.33,307.10 194.25,307.10 198.17,307.10 202.08,307.10 206.00,307.10 209.92,307.10 213.83,307.10 217.75,307.10 221.67,307.10 225.58,307.10 229.50,307.10 233.42,307.10 237.33,307.10 241.25,307.10 245.17,307.10 249.08,307.10 253.00,307.10 256.92,307.10 260.83,307.10 264.75,307.10 268.67,307.10 272.58,307.10 276.50,307.10 280.42,307.10 284.33,307.10 288.25,307.10 292.17,307.10 296.08,307.10 300.00,307.10 303.92,307.10 307.83,307.10 311.75,307.10 315.67,307.10 319.58,307.10 323.50,307.10 327.42,307.10 331.33,307.10 335.25,307.10 339.17,307.10 343.08,307.10 347.00,307.10 350.92,307.10 354.83,307.10 358.75,307.10 362.67,307.10 366.58,307.10 370.50,307.10 374.42,307.10 378.33,307.10 382.25,307.10 386.17,307.10 390.08,307.10 394.00,307.10 397.92,307.10 401.83,307.10 405.75,307.10 409.67,307.10 413.58,307.10 417.50,307.10 421.42,307.10 425.33,307.10 429.25,307.10 433.17,307.10 437.08,307.10 441.00,307.10 444.92,307.10 448.83,307.10 452.75,307.10 456.67,307.10 460.58,307.10 464.50,307.10 468.42,307.10 472.33,307.10 476.25,307.10 480.17,307.10 484.08,307.10 488.00,307.10 491.92,307.10 495.83,307.10 499.75,307.10 503.67,307.10 507.58,307.10 511.50,307.10 515.42,307.10 519.33,307.10 523.25,307.10 527.17,307.10 531.08,307.10 535.00,307.10' fill='none' stroke='steelblue' stroke-width='1.2'/><polyline points='65.00,256.39 68.92,252.15 72.83,247.90 76.75,257.06 80.67,266.22 84.58,275.38 88.50,284.54 92.42,284.54 96.33,284.54 100.25,284.54 104.17,284.54 108.08,284.54 112.00,284.54 115.92,284.54 119.83,284.54 123.75,284.54 127.67,284.54 131.58,284.54 135.50,284.54 139.42,284.54 143.33,284.54 147.25,284.54 151.17,284.54 155.08,284.54 159.00,284.54 162.92,284.54 166.83,284.54 170.75,284.54 174.67,284.54 178.58,284.54 182.50,284.54 186.42,284.54 190.33,284.54 194.25,284.54 198.17,284.54 202.08,284.54 206.00,284.54 209.92,284.54 213.83,284.54 217.75,284.54 221.67,284.54 225.58,284.54 229.50,284.54 233.42,284.54 237.33,284.54 241.25,284.54 245.17,284.54 249.08,284.54 253.00,284.54 256.92,284.54 260.83,284.54 264.75,284.54 268.67,284.54 272.58,284.54 276.50,284.54 280.42,284.54 284.33,284.54 288.25,284.54 292.17,284.54 296.08,284.54 300.00,284.54 303.92,284.54 307.83,284.54 311.75,284.54 315.67,284.54 319.58,284.54 323.50,284.54 327.42,284.54 331.33,284.54 335.25,284.54 339.17,284.54 343.08,284.54 347.00,284.54 350.92,284.54 354.83,284.54 358.75,284.54 362.67,284.54 366.58,284.54 370.50,284.54 374.42,284.54 378.33,284.54 382.25,284.54 386.17,284.54 390.08,284.54 394.00,284.54 397.92,284.54 401.83,284.54 405.75,284.54 409.67,284.54 413.58,284.54 417.50,284.54 421.42,284.54 425.33,284.54 429.25,284.54 433.17,284.54 437.08,284.54 441.00,284.54 444.92,284.54 448.83,284.54 452.75,284.54 456.67,284.54 460.58,284.54 464.50,284.54 468.42,284.54 472.33,284.54 476.25,284.54 480.17,284.54 484.08,284.54 488.00,284.54 491.92,284.54 495.83,284.54 499.75,284.54 503.67,284.54 507.58,284.54 511.50,284.54 515.42,284.54 519.33,284.54 523.25,284.54 527.17,284.54 531.08,284.54 535.00,284.54' fill='none' stroke='darkorange' stroke-width='1.2'/><polyline points='65.00,256.39 68.92,265.55 72.83,288.11 76.75,283.87 80.67,279.62 84.58,275.38 88.50,284.54 92.42,280.29 96.33,289.45 100.25,298.61 104.17,298.61 108.08,298.61 112.00,298.61 115.92,298.61 119.83,298.61 123.75,298.61 127.67,298.61 131.58,298.61 135.50,298.61 139.42,298.61 143.33,298.61 147.25,298.61 151.17,298.61 155.08,298.61 159.00,298.61 162.92,298.61 166.83,298.61 170.75,298.61 174.67,298.61 178.58,298.61 182.50,298.61 1

### Lecture du résultat

Le pooling écrémé **se ruine sûrement** : sa prime par tête (0,019) ne couvre
pas la sinistralité de ce qui lui reste (0,04) — le drift mensuel est
négatif, aucune trajectoire ne s'échappe. La spirale d'Akerlof est ici un
**taux de fuite du capital**. Le séparateur survit majoritairement — mais
remarquez que sa fréquence de ruine **ne s'annule pas** : même break-even par
type avec chargement, le risque Poisson résiduel met le surplus sous zéro de
temps en temps. C'est exactement ce que quantifie l'inégalité de Lundberg
(tranche T4) : $\psi(u) \lesssim e^{-\kappa u}$ — la probabilité de ruine
décroît **exponentiellement** avec le capital $u$ et le chargement $\eta$.
La non-existence d'équilibre Rothschild–Stiglitz de 17b a ainsi un prix
comptable : c'est la probabilité de ruine du dernier assureur à avoir cru au
pooling.

## §5 La valeur de l'information : le signal de Spence, lu par l'assureur

En 17b, le signal de Spence est **coûteux pour l'agent** — il sépare les types
parce que son coût est corrélé au type. L'assureur dispose d'un autre canal :
le **questionnaire de souscription** — un test imparfait, gratuit pour
l'agent, payant pour la compagnie.

Cadre : la prime $\beta$ est fixée par le marché au prix moyen
$\bar\theta$ (concurrence, pas de discrimination directe autorisée). Sans
test, l'assureur ne peut rien distinguer : accepter tout le monde rapporte
$\bar\theta - \bar\theta = 0$ en espérance. Avec le test, il décide
**d'accepter ou de refuser chaque segment** : accepter le segment $g$ rapporte
$\bar\theta - \lambda_g$ (son coût espéré posterior) si positif, sinon on
le refuse. La valeur du questionnaire est donc une **valeur d'échantillonnage
d'information** exactement comme en théorie de la décision :

$$\text{EVSI} = \sum_g P(g)\max(\bar\theta - \lambda_g,\, 0), \qquad
\text{EVPI} = \sum_t P(t)\max(\bar\theta - \theta_t,\, 0),$$

où l'EVPI (test parfait) majore toute information possible sur le type.

In [6]:
// Postérieurs d'un test binaire imparfait.
(double PPlus, double HPlus, double HMinus) TestPosteriors(double piHigh, double sens, double spec)
{
    double pPlus = piHigh * sens + (1 - piHigh) * (1 - spec);
    return (pPlus, piHigh * sens / pPlus, piHigh * (1 - sens) / (1 - pPlus));
}
double SegmentLambda(double piHPost) => piHPost * THETA_HIGH + (1 - piHPost) * THETA_LOW;
(double Premium, double LambdaPlus, double LambdaMinus, double ProfitPlus, double ProfitMinus, double Evsi, double Evpi, double Rendement) UnderwritingEvsi(double piHigh, double sens, double spec)
{
    double premium = SegmentLambda(piHigh); // prime concurrentielle theta_bar
    var post = TestPosteriors(piHigh, sens, spec);
    double lambdaPlus = SegmentLambda(post.HPlus), lambdaMinus = SegmentLambda(post.HMinus);
    double profitPlus = Math.Max(premium - lambdaPlus, 0.0), profitMinus = Math.Max(premium - lambdaMinus, 0.0);
    double evsi = post.PPlus * profitPlus + (1 - post.PPlus) * profitMinus;
    // EVPI : test parfait, chaque type accepte ssi theta_t < prime.
    double evpi = (1 - piHigh) * Math.Max(premium - THETA_LOW, 0.0) + piHigh * Math.Max(premium - THETA_HIGH, 0.0);
    return (premium, lambdaPlus, lambdaMinus, profitPlus, profitMinus, evsi, evpi, evpi > 0 ? evsi / evpi : double.NaN);
}
const double SENS = 0.70, SPEC = 0.80;
var posteriors = TestPosteriors(PI_HIGH, SENS, SPEC);
var res = UnderwritingEvsi(PI_HIGH, SENS, SPEC);
Verifier("postérieurs", posteriors.PPlus, 0.35, 1e-12);
Verifier("postérieurs", posteriors.HPlus, 0.6, 1e-12);
Verifier("postérieurs", posteriors.HMinus, 0.09 / 0.65, 1e-12);
if (!(res.ProfitPlus == 0 && res.ProfitMinus > 0 && res.Evsi <= res.Evpi)) throw new Exception("Décisions ou borne EVPI incorrectes");
var infoRows = new[]
{
    ("prime concurrentielle β = θ̄", res.Premium, 0.01900, 0.000005),
    ("coût espéré du segment + (λ₊)", res.LambdaPlus, 0.02800, 0.000005),
    ("coût espéré du segment − (λ₋)", res.LambdaMinus, 0.01415, 0.000005),
    ("profit sans test (accepter tout)", res.Premium - SegmentLambda(PI_HIGH), 0.0, 1e-12),
    ("EVSI du questionnaire", res.Evsi, 0.00315, 0.000005),
    ("EVPI (test parfait, plafond)", res.Evpi, 0.00630, 0.000005),
    ("rendement du test (EVSI/EVPI)", res.Rendement, 0.50000, 0.0005)
};
Console.WriteLine("quantité | valeur");
foreach (var (label, value, expected, tolerance) in infoRows)
{
    Verifier("information", value, expected, tolerance);
    Console.WriteLine($"{label} | {value:F5}");
}

quantité | valeur


prime concurrentielle β = θ̄ | 0,01900


coût espéré du segment + (λ₊) | 0,02800


coût espéré du segment − (λ₋) | 0,01415


profit sans test (accepter tout) | 0,00000


EVSI du questionnaire | 0,00315


EVPI (test parfait, plafond) | 0,00630


rendement du test (EVSI/EVPI) | 0,50000


### Lecture du résultat

Le segment $+$ a un coût espéré supérieur à la prime : l'assureur le
**refuse** ; le segment $-$ est rentable et accepté. Le questionnaire capte
ainsi une fraction de la valeur du test parfait — le **rendement EVSI/EVPI**
lu dans la table — et cette valeur est exactement le **prix plafond** de
l'information : frais de souscription, télématique ou examen médical au-delà,
l'opération détruit de la valeur. Le signal de Spence et le questionnaire
sont les deux faces symétriques de l'économie de l'information : l'un fait
payer l'agent pour **émettre**, l'autre fait payer la compagnie pour
**observer**. La tranche T5 de la série DecisionTheory calcule l'EVSI d'un questionnaire
complet avec calibration bayésienne du posterior ; ici le calcul fermé fixe
l'ordre de grandeur.

## Synthèse : la double lecture des quatre modèles

| Modèle (17b) | Lecture marché | Lecture assureur (ce carnet) |
|---|---|---|
| **Akerlof** (point fixe de participation) | le marché se ferme sous un seuil de qualité | spirale tarifaire : le pooling au prix moyen attire les H et fuit — §4 |
| **Spence** (signal coûteux) | le signal sépare les types car son coût est corrélé au type | questionnaire de souscription : la même séparation, achetée par l'assureur — EVSI, §5 |
| **Rothschild–Stiglitz** (menu séparateur) | équilibre concurrentiel break-even par type | tarification par type = crédibilité instantanée (Z = 1) ; son chargement est plafonné par la participation — §2-§3 |
| **Wilson/Miyazaki** (anticipation du retrait) | équilibre stable par anticipation | pooling **anticipé** : mutualisation assumée, K choisi — la crédibilité est le chemin entre pooling et séparation — §3 |

Le pont est complet : **le même objet** (le contrat $(\alpha, \beta)$) est un
échange d'utilité côté marché et un couple prime/sinistre côté bilan, et
chaque propriété d'équilibre se traduit en exigence de gestion — chargement,
crédibilité, capital, information. La formalisation Lean de la cohérence d'un
barème (tranche T6 de la série DecisionTheory : un barème incohérent expose l'assureur à un
arbitrage de tarification, une perte sûre) ferme la boucle côté preuve.

## Exercice 1 — chargement maximal du demi-couvert

Le chargement maximal $\eta^*$ dépend de la couverture $\alpha$. Calculer le
chargement maximal du type H pour une **demi-couverture** ($\alpha = 0{,}5$)
et comparer à la pleine couverture : la tolérance au chargement augmente-t-elle
ou diminue-t-elle quand la couverture baisse ? Pourquoi ?

In [7]:
// Exercice 1 — chargement maximal du demi-couvert
// TODO etudiant
 double? ExerciceChargementDemiCouverture()
{
    return null;
}

## Exercice 2 — années pour une crédibilité de 0.9 avec un mix plus fin

Avec $\pi_H = 0{,}10$ (marché plus sûr), recalculer $K$ et le nombre d'années
$n^{*}$ telles que $Z \geq 0{,}9$. Le chemin vers la séparation
s'allonge-t-il ou se raccourcit-il quand le marché devient plus homogène ?
Interpréter en termes d'antisélection.

In [8]:
// Exercice 2 — crédibilité avec un mix plus fin
// TODO etudiant
 double? ExerciceCredibiliteMix10()
{
    return null;
}

## Exercice 3 — re-tarifer le pool écrémé pour survivre

Après l'écrémage de §4, le portefeuille restant ne contient que des H. L'
assureur peut le re-tarifer à sa vraie sinistralité : prime par tête
$\theta_H$, chargement $\eta$. Par simulation Monte-Carlo (réutiliser
`SimulatePortfolio` avec `premiumPerHead = THETA_HIGH` et
`claimRatePerHead = THETA_HIGH`), trouver le plus petit $\eta$ (au pas de
1%) tel que la fréquence de ruine passe sous **5%**. Vérifier que ce $\eta$
reste sous le plafond de participation $\eta^*_H$ de §2.

In [9]:
// Exercice 3 — re-tarifer le pool écrémé
// TODO etudiant
 double? ExerciceEtaAntiRuine()
{
    return null;
}

## Exercice 4 — la valeur d'un test meilleur marché

Le questionnaire de §5 a une sensibilité de 0,70 et une spécificité de 0,80.
Une télématique plus chère offre 0,90 / 0,90. Calculer l'EVSI du test
amélioré et la part de l'EVPI qu'il capte. Que peut-on en déduire sur le
**prix plafond** de la télématique par contrat ?

In [10]:
// Exercice 4 — la valeur d'un test meilleur marché
// TODO etudiant
double? ExerciceEvsiTelematique()
{
    return null;
}

## Conclusion

Les quatre modèles d'information asymétrique de 17b ne décrivent pas seulement
des équilibres de marché : ils décrivent **les contraintes du métier
d'assureur**. Le chargement plafonné par la participation, la crédibilité
comme chemin temporel vers la séparation, la ruine comme prix comptable de
l'échec d'équilibre, l'EVSI comme budget d'information — chaque concept de
théorie des jeux a un double exact en gestion du risque, et réciproquement.

**Pour aller plus loin** : la série *DecisionTheory* développe
chaque jambe de ce pont en profondeur — décomposition de prime (T1),
fréquence × sévérité et pooling hiérarchique (T2), crédibilité
Bühlmann–Straub avec calibration PyMC (T3), ruine de Cramér–Lundberg et
capital réglementaire (T4), EVPI/EVSI d'un questionnaire de souscription (T5),
et la cohérence d'un barème de primes prouvée en Lean (T6) : un barème qui
viole l'inclusion–exclusion expose l'assureur à un arbitrage de tarification,
une perte stricte et sûre.

### Parité numérique

Les assertions des sections contrat, chargement, crédibilité et information comparent les calculs aux références, avec les tolérances déclarées dans le code ; leurs écarts maximaux sont conservés dans `ecarts`. Les écarts de simulation sont affichés dans la section ruine : les trajectoires .NET ne sont pas les tirages NumPy et ne constituent donc pas une reproduction de ceux-ci.